# HrudayAI — Milestone 2: Open-Source Symptom Extraction (Hindi & English)

This Google Colab CPU-compatible notebook loads Phase 1 transcription JSON (`phase1_voice/outputs/transcription_results.json`) and extracts structured symptoms (`canonical_name`, `original_term`, `duration`, `severity`, `negated`) using free, open-source local Python tools.

- **Languages:** Hindi (`hi`) and English (`en`)
- **External / Paid APIs:** None
- **Disease Prediction:** Not connected in this milestone

## 1. Environment & Dependency Setup (Colab CPU)

In [ ]:
import importlib
import json
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PUBLIC_REPO_URL = "https://github.com/itsakshatsinghh/HrudayAI.git"
COLAB_REPO_ROOT = Path("/content/HrudayAI")

freshly_cloned = False
if IN_COLAB:
    if not COLAB_REPO_ROOT.exists():
        try:
            subprocess.run(
                ["git", "clone", "--branch", "main", PUBLIC_REPO_URL, str(COLAB_REPO_ROOT)],
                check=True, capture_output=True, text=True,
            )
            freshly_cloned = True
        except subprocess.CalledProcessError as exc:
            raise RuntimeError(
                f"Failed to clone public repo {PUBLIC_REPO_URL}: {exc.stderr or exc}"
            ) from exc
    elif not (COLAB_REPO_ROOT / ".git").exists():
        raise RuntimeError(
            f"Directory {COLAB_REPO_ROOT} exists but is not a Git repository. "
            f"Remove it with `!rm -rf {COLAB_REPO_ROOT}` and re-run this cell."
        )
    if str(COLAB_REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(COLAB_REPO_ROOT))
else:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "shared" / "colab_bootstrap.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            break

from shared.colab_bootstrap import bootstrap_repo

PULL_LATEST = True
bootstrap_info = bootstrap_repo(pull_latest=(PULL_LATEST and not freshly_cloned))
REPO_ROOT = bootstrap_info["repo_root"]

REQ_FILE = REPO_ROOT / "phase2_symptom_extraction" / "requirements.txt"
if IN_COLAB and REQ_FILE.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ_FILE)])
    importlib.invalidate_caches()

print(f"Working dir      : {Path.cwd()}")
print(f"rapidfuzz ready  : {importlib.util.find_spec('rapidfuzz') is not None}")

## 2. Load Phase 1 Transcription JSON (`phase1_voice/outputs/transcription_results.json`)

> **Colab Runtime Persistence Note:** Colab runtime storage is temporary and resets when a session disconnects. `phase1_voice/outputs/transcription_results.json` will not persist across fresh runtimes.

Choose how to supply input:
1. **Default (`phase1_voice/outputs/transcription_results.json`):** Loads the Phase 1 output file if present in this runtime.
2. **Upload in Colab (`UPLOAD_PHASE1_JSON = True`):** Upload a saved `transcription_results.json` from your computer/Drive.
3. **Demonstration Only (`USE_DEMO_SAMPLE = True`):** Explicitly runs a labeled demonstration sample (never reported as real patient transcription).

In [ ]:
PHASE1_OUTPUT_PATH = REPO_ROOT / "phase1_voice" / "outputs" / "transcription_results.json"
UPLOAD_PHASE1_JSON = False
USE_DEMO_SAMPLE = False

phase1_payload = None
INPUT_SOURCE_LABEL = "none"

if PHASE1_OUTPUT_PATH.exists():
    phase1_payload = json.loads(PHASE1_OUTPUT_PATH.read_text(encoding="utf-8"))
    INPUT_SOURCE_LABEL = f"REAL_PHASE1_FILE ({PHASE1_OUTPUT_PATH})"
    print(f"Loaded Phase 1 output from: {PHASE1_OUTPUT_PATH}")
elif IN_COLAB and UPLOAD_PHASE1_JSON:
    from google.colab import files  # type: ignore

    print("Upload your saved Phase 1 `transcription_results.json` file:")
    uploaded = files.upload()
    if uploaded:
        fname = next(iter(uploaded.keys()))
        PHASE1_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
        PHASE1_OUTPUT_PATH.write_bytes(uploaded[fname])
        phase1_payload = json.loads(PHASE1_OUTPUT_PATH.read_text(encoding="utf-8"))
        INPUT_SOURCE_LABEL = f"UPLOADED_PHASE1_FILE ({fname})"
        print(f"Restored Phase 1 output to {PHASE1_OUTPUT_PATH}")
elif USE_DEMO_SAMPLE:
    INPUT_SOURCE_LABEL = "DEMONSTRATION_SAMPLE_ONLY (NOT REAL PATIENT TRANSCRIPTION)"
    print(f"WARNING: {INPUT_SOURCE_LABEL}")
    phase1_payload = [
        {
            "detected_language": "hi",
            "text": "मुजे कल से बुखार है, सिर में दर्द हो रहा है, और तोडी कमजोरी भी महसुस हो रही है",
        },
        {
            "detected_language": "en",
            "text": "I have been experiencing fevers since yesterday along with a headache and some weaknesses.",
        },
    ]
else:
    print(f"Phase 1 output not found at: {PHASE1_OUTPUT_PATH}")
    print("Colab runtime storage is temporary—files from previous sessions do not persist automatically.")
    print("To continue, choose one of:")
    print("  1. Run Phase 1 (`phase1_voice/notebooks/01_audio_to_text.ipynb`) in this runtime to generate transcription_results.json.")
    print("  2. Set `UPLOAD_PHASE1_JSON = True` above and re-run this cell to upload your saved JSON.")
    print("  3. Set `USE_DEMO_SAMPLE = True` above ONLY if you want to run a clearly labeled demonstration sample.")

if phase1_payload is not None:
    print(f"Input source: {INPUT_SOURCE_LABEL}")
    print(json.dumps(phase1_payload, indent=2, ensure_ascii=False))

## 3. Run Structured Symptom Extraction on Phase 1 Output

In [ ]:
from phase2_symptom_extraction.src import extract_from_phase1_payload, extract_symptoms

if phase1_payload is None:
    print("Skipping extraction: no Phase 1 JSON loaded. Restore/upload `transcription_results.json` or enable `USE_DEMO_SAMPLE = True` in Section 2.")
else:
    extracted_results = extract_from_phase1_payload(phase1_payload)
    print(f"=== Structured Symptom Extraction Output [{INPUT_SOURCE_LABEL}] ===")
    print(json.dumps(extracted_results, indent=2, ensure_ascii=False))

## 4. Milestone 2.5 Clinical Language Validation Cases (Hindi & English)

Demonstrates negation, historical vs. current symptoms, severity, uncertainty, questions, duration, ambiguity, and past-to-current temporal transitions.

In [ ]:
sample_cases = [
    ("en", "I don't have fever, but I have severe headache for three days."),
    ("hi", "मुझे दो दिन से तेज बुखार है और सिर में दर्द नहीं है।"),
    ("en", "I had fever last week, but I don't have it now."),
    ("hi", "कल बुखार था, लेकिन अब नहीं है।"),
    ("en", "I think I might have a fever."),
    ("hi", "शायद मुझे बुखार है।"),
    ("hi", "पता नहीं बुखार है या नहीं।"),
]

for lang, text in sample_cases:
    out = extract_symptoms(text, language=lang)
    print(f"\n--- Clinical Language Case ({lang}) ---")
    print(json.dumps(out, indent=2, ensure_ascii=False))

## 5. Run Unit Tests Inside Colab

In [ ]:
import unittest

suite = unittest.defaultTestLoader.discover(str(REPO_ROOT / "phase2_symptom_extraction" / "tests"))
runner = unittest.TextTestRunner(verbosity=2)
test_result = runner.run(suite)
assert test_result.wasSuccessful(), "Phase 2 symptom extraction tests failed!"